<a href="https://colab.research.google.com/github/sittana-afifi/KTT-Fellow-Hackathon---G1---T2.1-Compressed-Crop-Disease-Classifier/blob/main/app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from fastapi import FastAPI, UploadFile, File
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import io

app = FastAPI()

# --- 1. MODEL SETUP (Matches your fixed architecture) ---
model = models.mobilenet_v3_small(weights=None)
model.classifier = nn.Sequential(
    nn.Linear(576, 1024),
    nn.Hardswish(inplace=True),
    nn.Dropout(p=0.2, inplace=True),
    nn.Linear(1024, 5)
)
model.load_state_dict(torch.load('models/hinga_model.pth', map_location='cpu'))
model.eval()

class_names = ['Bean Spot', 'Cassava Mosaic', 'Maize Blight', 'Healthy Maize', 'Maize Rust']

# --- 2. PREPROCESSING ---
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    # Read image
    data = await file.read()
    image = Image.open(io.BytesIO(data)).convert('RGB')

    # Process
    input_tensor = transform(image).unsqueeze(0)

    # Inference
    with torch.no_grad():
        output = model(input_tensor)
        prob = torch.nn.functional.softmax(output, dim=1)
        conf, pred_idx = torch.max(prob, 1)

    # Return USSD-friendly JSON
    return {
        "text_response": f"AI Diagnosis: {class_names[pred_idx]}\nConfidence: {conf.item()*100:.1f}%\nRecommendation: Apply recommended fungicide.",
        "disease": class_names[pred_idx]
    }

In [ ]:
!pip install fastapi uvicorn python-multipart
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏
added 22 packages in 3s
⠏
⠏3 packages are looking for funding
⠏  run `npm fund` for details
⠏

In [ ]:
# This will output a URL. Use that for your USSD tests.
!ssh -o StrictHostKeyChecking=no -R 80:localhost:8000 serveo.net

Forwarding HTTP traffic from https://5df6f5bf602f67b6-35-201-131-94.serveousercontent.com


In [ ]:
import requests

# 1. Your Live Tunnel URL
url = "https://honest-paws-carry.loca.lt/predict"

# 2. Path to a sample image in your Colab
image_path = "/content/ktt/data/mini_plant_set/maize_rust/49339.jpg"

# 3. Bypass the Localtunnel "Reminder" screen
headers = {"Bypass-Tunnel-Reminder": "true"}

# 4. Fire the request!
with open(image_path, "rb") as f:
    files = {"file": f}
    response = requests.post(url, files=files, headers=headers)

if response.status_code == 200:
    print("✨ USSD API VERIFIED!")
    print("----------------------------")
    print(response.json()['ussd_message'])
    print("----------------------------")
else:
    print(f"❌ Error: {response.status_code}")
    print(response.text)

❌ Error: 503
503 - Tunnel Unavailable
